In [25]:
import pandas as pd
import numpy as np
from collections import defaultdict
matches = pd.read_csv('../data/processed/features_v5.csv')

# Make sure data is chronological
matches['MatchDateTime'] = pd.to_datetime(matches['MatchDateTime'])
matches = matches.sort_values('MatchDateTime').reset_index(drop=True)

In [26]:
home_xg_for = []
away_xg_for = []

home_xg_against = []
away_xg_against = []
season_history = defaultdict(lambda: defaultdict(list))

In [27]:
def calculate_season_stats(previous_matches, team):
    if len(previous_matches) == 0:
        return {
            'games': 0,
            'xg_for': np.nan,
            'xg_against': np.nan
        }

    xg_for = []
    xg_against = []

    for match in previous_matches:

        # Team was home
        if match['HomeTeam'] == team:
            team_xg = match['HomeXG']
            opponent_xg = match['AwayXG']

        # Team was away
        else:
            team_xg = match['AwayXG']
            opponent_xg = match['HomeXG']

        # Only use matches where BOTH xG values exist
        if pd.notna(team_xg) and pd.notna(opponent_xg):
            xg_for.append(team_xg)
            xg_against.append(opponent_xg)

    # No usable xG data yet
    if len(xg_for) == 0:
        return {
            'games': 0,
            'xg_for': 0,
            'xg_against': 0
        }

    return {
        'games': len(xg_for),
        'xg_for': sum(xg_for) / len(xg_for),
        'xg_against': sum(xg_against) / len(xg_against)
    }

In [28]:
for _, current_match in matches.iterrows():

    season = current_match['Season']
    home_team = current_match['HomeTeam']
    away_team = current_match['AwayTeam']

    # Previous matches in the SAME season
    home_previous = season_history[season][home_team]
    away_previous = season_history[season][away_team]

    # Calculate stats BEFORE current match
    home_stats = calculate_season_stats(home_previous, home_team)
    away_stats = calculate_season_stats(away_previous, away_team)

    # Store home features
    home_xg_for.append(home_stats['xg_for'])
    home_xg_against.append(home_stats['xg_against'])

    # Store away features
    away_xg_for.append(away_stats['xg_for'])
    away_xg_against.append(away_stats['xg_against'])

    # Add current match to history AFTER calculating features
    season_history[season][home_team].append(current_match)
    season_history[season][away_team].append(current_match)

In [29]:
matches['HomeXGPerGame'] = home_xg_for
matches['AwayXGPerGame'] = away_xg_for

matches['HomeXGA_PerGame'] = home_xg_against
matches['AwayXGA_PerGame'] = away_xg_against

C:\Users\harry\AppData\Local\Temp\ipykernel_3860\1961538097.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['HomeXGPerGame'] = home_xg_for
C:\Users\harry\AppData\Local\Temp\ipykernel_3860\1961538097.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['AwayXGPerGame'] = away_xg_for
C:\Users\harry\AppData\Local\Temp\ipykernel_3860\1961538097.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining al

In [ ]:
matches['XGForDiffPg'] = (
    matches['HomeXGPerGame'] -
    matches['AwayXGPerGame']
)

matches['XGAgainstDiffPg'] = (
    matches['AwayXGA_PerGame'] -
    matches['HomeXGA_PerGame']
)

C:\Users\harry\AppData\Local\Temp\ipykernel_3860\2686623778.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['XgFDiffPg'] = (
C:\Users\harry\AppData\Local\Temp\ipykernel_3860\2686623778.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['XgADiffPg'] = (


In [31]:
matches.tail(20)[['HomeTeam', 'AwayTeam', 'HomeXG', 'AwayXG', 'XgFDiffPg', 'XgADiffPg']]

,HomeTeam,AwayTeam,HomeXG,AwayXG,XgFDiffPg,XgADiffPg
1880,Aston Villa,Liverpool,2.525200,1.550270,-0.238173,-0.213209
1881,Man United,Nott'm Forest,NaN,NaN,0.000000,0.000000
1882,Brentford,Crystal Palace,1.545640,1.822740,0.038172,0.116566
1883,Everton,Sunderland,1.321810,1.283830,0.267635,0.052113
1884,Leeds,Brighton,0.747756,2.232380,-0.026405,0.056746
1885,Wolves,Fulham,NaN,NaN,-1.246345,1.706780
1886,Newcastle,West Ham,NaN,NaN,-1.244493,1.858007
1887,Arsenal,Burnley,1.226950,0.194884,1.128498,1.189877
1888,Bournemouth,Man City,NaN,NaN,1.814812,-1.480237
1889,Chelsea,Tottenham,1.092620,2.152270,0.695158,-0.042335


In [32]:
matches['HomeXGDPerGame'] = (
    matches['HomeXGPerGame'] -
    matches['HomeXGA_PerGame']
)

matches['AwayXGDPerGame'] = (
    matches['AwayXGPerGame'] -
    matches['AwayXGA_PerGame']
)

matches['XGDDiff'] = (
    matches['HomeXGDPerGame'] -
    matches['AwayXGDPerGame']
)

C:\Users\harry\AppData\Local\Temp\ipykernel_3860\1778873391.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['HomeXGDPerGame'] = (
C:\Users\harry\AppData\Local\Temp\ipykernel_3860\1778873391.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['AwayXGDPerGame'] = (
C:\Users\harry\AppData\Local\Temp\ipykernel_3860\1778873391.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once

In [33]:
def calculate_team_stats(last5, team):
    stats = {
        "XGForLast5": 0,
        "XGAgainstLast5": 0
    }

    if not last5:
        return stats

    xgf = []
    xga = []

    for match in last5:
        if match["HomeTeam"] == team:
            xgf.append(match["HomeXG"])
            xga.append(match["AwayXG"])

        else:  # team was away
            xgf.append(match["AwayXG"])
            xga.append(match["HomeXG"])

    stats["XGForLast5"] = sum(xgf) / len(xgf)
    stats["XGAgainstLast5"] = sum(xga) / len(xga)
    return stats

In [34]:

# Store feature columns automatically
features = defaultdict(list)
team_history = defaultdict(list)

for _, current_match in matches.iterrows():

    home_team = current_match["HomeTeam"]
    away_team = current_match["AwayTeam"]

    home_last5 = team_history[home_team][-5:]
    away_last5 = team_history[away_team][-5:]

    home_stats = calculate_team_stats(home_last5, home_team)
    away_stats = calculate_team_stats(away_last5, away_team)

    for key, value in home_stats.items():
        features[f"Home{key}"].append(value)

    for key, value in away_stats.items():
        features[f"Away{key}"].append(value)

    # Update history AFTER calculating features
    team_history[home_team].append(current_match)
    team_history[away_team].append(current_match)

# Add all features to dataframe
for col, values in features.items():
    matches[col] = values

C:\Users\harry\AppData\Local\Temp\ipykernel_3860\1207791489.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches[col] = values


In [35]:
matches['XGForDiffLast5'] = (
    matches['HomeXGForLast5'] -
    matches['AwayXGForLast5']
)

matches['XGAgainstDiffLast5'] = (
    matches['AwayXGAgainstLast5'] -
    matches['HomeXGAgainstLast5']
)

C:\Users\harry\AppData\Local\Temp\ipykernel_3860\542012244.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['XGForDiffLast5'] = (
C:\Users\harry\AppData\Local\Temp\ipykernel_3860\542012244.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['XGAgainstDiffLast5'] = (


In [36]:
matches['HomeXGDLast5'] = (
    matches['HomeXGForLast5'] -
    matches['HomeXGAgainstLast5']
)

matches['AwayXGDLast5'] = (
    matches['AwayXGForLast5'] -
    matches['AwayXGAgainstLast5']
)

matches['XGDDiffLast5'] = (
    matches['HomeXGDLast5'] -
    matches['AwayXGDLast5']
)

C:\Users\harry\AppData\Local\Temp\ipykernel_3860\1613299282.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['HomeXGDLast5'] = (
C:\Users\harry\AppData\Local\Temp\ipykernel_3860\1613299282.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['AwayXGDLast5'] = (
C:\Users\harry\AppData\Local\Temp\ipykernel_3860\1613299282.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once usi

In [37]:
matches.to_csv('../data/processed/features_v6.csv', index=False)

print('Saved features_v6.csv')

Saved features_v6.csv
